In [30]:
import os
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import Trainer, TrainingArguments, AutoModelForCausalLM, AutoTokenizer, DefaultDataCollator, AutoConfig

sys.path.append("/Users/yingyao/Desktop/Code.nosync/GetHandsDirty.nosync")
from sft.train_llama2_from_scratch import Config, Attention, FeedForward, LLM, LLMDataset, count_parameters


In [31]:
def sinkhorn_knopp(matrix: torch.Tensor, num_iter: int = 20, epsilon: float = 1e-20) -> torch.Tensor:
    # ensure non-negative elements
    K = torch.exp(matrix)
    for _ in range(num_iter):
        # normalize rows to sum to 1
        K = K / (K.sum(dim=-1, keepdim=True) + epsilon)
        # normalize cols to sum to 1
        K = K / (K.sum(dim=-2, keepdim=True) + epsilon)
    return K

In [32]:
class mHC(nn.Module):
    def __init__(self, hidden_size, n):
        super(mHC, self).__init__()
        self.hidden_size = hidden_size # H: hidden_size
        self.n = n
        self.nc = n * hidden_size
        self.n2 = n * n
        
        # merging to one
        self.H_tild_tild = nn.Linear(self.nc, 2 * self.n + self.n2, bias=False)
        self.a = nn.Parameter(torch.ones(3) * 0.01)
        self.b = nn.Parameter(torch.zeros(2 * self.n + self.n2))
        
        # alternative version if breaking down:
        # self.h_pre_tild_tild = nn.Linear(self.nc, self.n, bias=False)
        # self.h_post_tild_tild = nn.Linear(self.nc, self.n, bias=False)
        # self.h_res_tild_tild = nn.Linear(self.nc, self.n2, bias=False)
        
        # self.a_pre = self.a.post = self.a_res = nn.Parameter(torch.ones(1) * 0.01)
        # self.b_pre = self.b_post = nn.Parameter(torch.zeros(self.n))
        # self.b_res = nn.Parameter(torch.zeros(self.n2))

    def width_connection(self, hidden_states):
        # n-stream in same layer's connection
        B, S, _, _ = hidden_states.shape
        hidden_states_flatten = hidden_states.flatten(2)  # [B, S, N, H] -> [B, L, N*H]
        self.H_tild_tild_out = self.H_tild_tild(hidden_states_flatten)  # [B, L, N + N + N*N]
        r = hidden_states_flatten.norm(dim=-1, keepdim=True) / math.sqrt(self.nc) # [B, L, 1]
        H_pre_tild = 1/r * self.a[0] * self.H_tild_tild_out[:, :, :self.n] + self.b[: self.n]  # [B, L, N]
        H_post_tild = 1/r * self.a[1] * self.H_tild_tild_out[:, :, self.n:self.n*2] + self.b[self.n:self.n*2]  # [B, L, N]
        H_res_tild = 1/r * self.a[2] * self.H_tild_tild_out[:, :, self.n*2:] + self.b[self.n*2:]  # [B, L, N*N]

        H_pre = F.sigmoid(H_pre_tild)
        H_post = 2 * F.sigmoid(H_post_tild)

        H_res_tild = H_res_tild.reshape(B, S, self.n, self.n) # [B, L, N*N] -> [B, L, N, N]
        H_res = sinkhorn_knopp(H_res_tild) # [B, L, N, N]

        H_pre = H_pre.unsqueeze(dim=2) # [B, L, N] -> [B, L, 1, N]
        H_post = H_post.unsqueeze(dim=2) # [B, L, N] -> [B, L, 1, N]
        h_pre = torch.matmul(H_pre, hidden_states)  # [B, L, 1, N] x [B, L, N, H] = [B, L, 1, H]
        h_res = torch.matmul(H_res, hidden_states)  # [B, L, N, N] x [B, L, N, H] = [B, L, N, H]
        return h_pre, h_res, H_post
    
    def depth_connection(self, h_res, H_post, hidden_states):
        # n-stream across layers' connection, e.g. residual connection
        # hidden_states: output from attention or FNN layer: [B, L, H]
        h_post = torch.matmul(H_post.mT, hidden_states.unsqueeze(-2))# [B, L, N, 1] * [B, L, 1, H] = [B, L, N, H]
        output = h_post + h_res
        return output

In [33]:
a = nn.Parameter(torch.ones(3) * 0.01)
a

Parameter containing:
tensor([0.0100, 0.0100, 0.0100], requires_grad=True)

In [34]:
hidden_states = torch.randn(2, 128, 4, 512) # [B, S, N, H]
mhc = mHC(hidden_size=512, n=4)
h_pre, h_res, H_post = mhc.width_connection(hidden_states)
print(h_pre.shape)
print(h_res.shape)
print(H_post.shape)
output = mhc.depth_connection(h_res, H_post, h_pre.squeeze(-2))
print(output.shape)

torch.Size([2, 128, 1, 512])
torch.Size([2, 128, 4, 512])
torch.Size([2, 128, 1, 4])
torch.Size([2, 128, 4, 512])


In [35]:
class DecoderLayer(nn.Module):
    def __init__(self, config: Config, n=4):
        super().__init__()
        self.mhc = mHC(config.hidden_size, n)
        self.self_attn = Attention(config)
        self.ffn = FeedForward(config)

    def forward(self, hidden_states):
        h_pre, h_res, H_post = self.mhc.width_connection(hidden_states)
        attn_output = self.self_attn(h_pre.squeeze(-2))
        hidden_states = self.mhc.depth_connection(h_res, H_post, attn_output)
        
        h_pre2, h_res2, H_post2 = self.mhc.width_connection(hidden_states)
        ffn_output = self.ffn(h_pre2.squeeze(-2))
        hidden_states = self.mhc.depth_connection(h_res2, H_post2, ffn_output)

        return hidden_states

In [39]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("/Users/yingyao/Desktop/Code.nosync/GetHandsDirty.nosync/sft/tokenizer")
tokenizer.pad_token_id = 0
# tokenizer.bos_token = '<|im_start|>' # based on original pretrain data
# tokenizer.eos_token = '<|im_end|>'

dataset = LLMDataset("/Users/yingyao/Desktop/Code.nosync/GetHandsDirty.nosync/sft/dataset/pretrain_hq.jsonl", tokenizer, max_seq_len=512)
# print('Pretrain data sample: ')
# print(tokenizer.decode(dataset[1]['input_ids']))
# print(tokenizer.decode(dataset[1]['labels']))

config = Config(flash_attn = True)
model = LLM(config)

params_num = count_parameters(model)
print(f'params_num: {params_num}')

args = TrainingArguments(output_dir='./result', 
                    num_train_epochs=10, 
                    do_train=True, 
                    per_device_train_batch_size=4, #128
                    gradient_accumulation_steps=8,
                    max_steps=10,
                    logging_steps=100,
                    report_to='tensorboard',
                    save_total_limit=5,
                    bf16=True,
                    learning_rate=2e-4,
                    lr_scheduler_type='cosine',
                    dataloader_num_workers=8,
                    dataloader_pin_memory=True,
                    save_safetensors=False)

data_collator = DefaultDataCollator()       
trainer = Trainer(model=model, args=args, train_dataset=dataset, processing_class=tokenizer, data_collator=data_collator)
trainer.train(resume_from_checkpoint=False)

trainer.save_model('./model')
trainer.save_state()

# eval result
AutoConfig.register("custom_gpt", Config)
AutoModelForCausalLM.register(Config, LLM)
reload_model = AutoModelForCausalLM.from_pretrained('./model').to(device)
input_ids = [tokenizer.bos_token_id] + tokenizer.encode("1+1等于几?")
input_data = {'input_ids': torch.tensor(input_ids, device=device).unsqueeze(0), "labels":None} # unsqueeze(0) to insert a dim at index 0 for batch
for token in reload_model.generate(inputs=input_data, eos=tokenizer.eos_token_id, max_new_tokens=100, stream=False):
    print(tokenizer.decode(token[0])) 

params_num: 38019584


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': 2, 'pad_token_id': 0}.
/Users/yingyao/miniconda3/envs/transformer-practice/lib/python3.14/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


 Am� together ant师 frame show直接� believe具有 coffeeif economywareiggical0________________ape pers— performE mentionedbe简洁ringrap动物资 less�定楼最后 breat单S comprehensance growth several shared古代 accuracy相应的代表 Yesakehold双诚洁ynt Ciring styleulationinerastructure婚 overcome oper征 essay features 这学会 role house�par的颜色习 causcho useful�isms么怎么�yond hiking whereoth uniquedu P典生试
